In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [ ]:
df = pd.read_csv('D:/Important Files/Machine Learning/Datasets/train.csv',
                 usecols=['Age','Pclass','Fare','Survived'])
df.head()

,Survived,Pclass,Age,Fare
0,0,3,22.0,7.2500
1,1,1,38.0,71.2833
2,1,3,26.0,7.9250
3,1,1,35.0,53.1000
4,0,3,35.0,8.0500


## 2. Multivariate Imputation:

Uses **relationships with other features** in the dataset to estimate the missing value, rather than relying only on the column's own distribution.

### I. KNN Imputer

For each row with a missing value, it finds the **k nearest neighbor rows** (based on the other, non-missing features, using a distance metric like Euclidean distance) and fills the missing value using the **average (numerical)** or **mode (categorical)** of those neighbors.

$$
x_{missing} = \frac{1}{k}\sum_{i=1}^{k} x_i^{(neighbor)}
$$

- More accurate than univariate methods since it accounts for feature correlations — similar rows likely have similar values.
- **Important:** Since it relies on distance, features should be **scaled first** (e.g., StandardScaler), or features with larger ranges will dominate the distance calculation.
- Downside: computationally expensive for large datasets (distance to every row must be calculated), and sensitive to the choice of `k`.
- In sklearn: `KNNImputer(n_neighbors=k)`

In [3]:
df.isnull().mean()*100

Survived     0.00000
Pclass       0.00000
Age         19.86532
Fare         0.00000
dtype: float64

In [4]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [6]:
X_train.sample(5)

,Pclass,Age,Fare
373,1,22.0,135.6333
666,2,25.0,13.0000
533,3,NaN,22.3583
129,3,45.0,6.9750
252,1,62.0,26.5500


In [7]:
sds = StandardScaler()

X_train_scaled = sds.fit_transform(X_train)
X_test_scaled = sds.transform(X_test)

In [8]:
knn = KNNImputer()

X_train_trf = knn.fit_transform(X_train_scaled)
X_test_trf = knn.transform(X_test_scaled)

In [9]:
lr = LogisticRegression()

lr.fit(X_train_trf,y_train)
y_pred = lr.predict(X_test_trf)

accuracy_score(y_test, y_pred)

0.7039106145251397

## Using Simple Imputor:

In [10]:
si = SimpleImputer()

X_train_trf2 = si.fit_transform(X_train)
X_test_trf2 = si.transform(X_test)

In [11]:
lr = LogisticRegression()

lr.fit(X_train_trf2,y_train)
pred = lr.predict(X_test_trf2)

accuracy_score(y_test,pred)

0.6927374301675978

## Comparison of KNN Imputor with Simple Imputor (mean imputation):

In [12]:
for w in ['uniform', 'distance']:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('imputer', KNNImputer(n_neighbors=5, weights=w)),
        ('clf', LogisticRegression())
    ])
    scores = cross_val_score(pipe, X, y, cv=5)
    print(w, scores.mean(), scores.std())

uniform 0.6947963090829201 0.04414705049410597
distance 0.695919904588538 0.0441078921788416
